# TimeSage Complete Demo - Airline Passengers Dataset

> **The Wise Time Series Library** - Beautiful EDA, All Models, Plain-English Interpretation

This notebook demonstrates **every feature** of timesage-ts using the classic airline passengers dataset.

- Install: `pip install timesage-ts[full]`
- Docs: [GitHub](https://github.com/mlnjsh/timesage)
- PyPI: [timesage-ts](https://pypi.org/project/timesage-ts/)

## Installation

In [ ]:
!pip install timesage-ts[full] -q

## Import & Setup

In [ ]:
import timesage as ts
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

ts.hello()

## Load Airline Passengers Dataset

Classic Box-Jenkins dataset: monthly totals of international airline passengers (1949-1960), 144 observations.

In [ ]:
data = ts.load_airline()
print(f"Type: {type(data)}")
print(f"Shape: {data.shape}")
print(f"Date range: {data.index.min()} to {data.index.max()}")
data.head(10)

## Create TimeSeries Object

In [ ]:
series = ts.TimeSeries(data, name="Airline Passengers")
print(series)

## Basic Description

In [ ]:
series.describe()

## Plot Raw Series

In [ ]:
plt.figure(figsize=(12, 5))
series.plot()
plt.title("Airline Passengers - Raw Time Series", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Trend & Outlier Detection

In [ ]:
plt.figure(figsize=(12, 5))
series.plot(show_trend=True, show_outliers=True)
plt.title("Trend Line & Outlier Detection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Theme Gallery

TimeSage ships with 3 themes: **sage** (default), **dark**, **minimal**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, theme_name in zip(axes, ["sage", "dark", "minimal"]):
    ts.set_theme(theme_name)
    plt.sca(ax)
    series.plot()
    ax.set_title(f"Theme: {theme_name}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
ts.set_theme("sage")

## Stationarity Testing

Augmented Dickey-Fuller (ADF) and KPSS tests.

In [ ]:
series.test_stationarity()

## Seasonality Detection

ACF peak analysis for seasonal patterns.

In [ ]:
series.detect_seasonality()

## ACF & PACF

In [ ]:
series.plot_acf(lags=40)
plt.tight_layout()
plt.show()

## Time Series Decomposition

Trend + Seasonal + Residual components.

In [ ]:
series.decompose(model="additive", period=12)
series.plot_decomposition()
plt.tight_layout()
plt.show()

## Full EDA

One command for everything: description, stationarity, seasonality, and plots.

In [ ]:
series.eda(show_plots=True)

---
## Feature Engineering

Create ML features: lags, rolling stats, EWM, temporal features.

In [ ]:
features_df = series.create_features(
    lags=[1, 2, 3, 6, 12, 24],
    windows=[3, 6, 12],
    temporal=True
)
print(f"Features shape: {features_df.shape}")
print(f"Columns: {list(features_df.columns)}")
features_df.head()

### Feature Pipeline Direct Use

In [ ]:
from timesage.features import FeaturePipeline

pipeline = FeaturePipeline(
    lags=[1, 2, 3, 6, 12],
    windows=[3, 6, 12],
    temporal=True
)
feat_df = pipeline.transform(series.dataframe)
print(f"Total features: {feat_df.shape[1]}")
feat_df.dropna().head()

---
## Statistical Models

### ARIMA
Automatically selects best (p,d,q) via AIC grid search.

In [ ]:
result_arima = series.forecast(horizon=12, model="arima")
result_arima.summary()

In [ ]:
result_arima.plot()
plt.title("ARIMA Forecast", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### ETS (Holt-Winters)
Exponential smoothing with level, trend, and seasonality.

In [ ]:
result_ets = series.forecast(horizon=12, model="ets")
result_ets.summary()

In [ ]:
result_ets.plot()
plt.title("ETS Forecast", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Theta Method

In [ ]:
result_theta = series.forecast(horizon=12, model="theta")
result_theta.summary()

In [ ]:
result_theta.plot()
plt.title("Theta Forecast", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Machine Learning Models

### Random Forest

In [ ]:
result_rf = series.forecast(horizon=12, model="rf")
result_rf.summary()

In [ ]:
result_rf.plot()
plt.title("Random Forest Forecast", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### XGBoost

In [ ]:
result_xgb = series.forecast(horizon=12, model="xgboost")
result_xgb.summary()

In [ ]:
result_xgb.plot()
plt.title("XGBoost Forecast", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### LightGBM

In [ ]:
result_lgb = series.forecast(horizon=12, model="lightgbm")
result_lgb.summary()

In [ ]:
result_lgb.plot()
plt.title("LightGBM Forecast", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## AutoML - Automatic Model Selection

TimeSage trains ARIMA, ETS, Theta, RF on validation, picks the winner by lowest MAPE.

In [ ]:
result_auto = series.forecast(horizon=12, model="auto")
result_auto.summary()

In [ ]:
result_auto.plot()
plt.title("Auto Model Selection - Best Forecast", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Plain-English Interpretation

Understand your forecast without a statistics degree.

In [ ]:
result_auto.interpret()

### Detailed Interpretation Breakdown

In [ ]:
from timesage.interpret.explainer import explain_forecast

explanation = explain_forecast(result_auto)
for key, value in explanation.items():
    sep = "=" * 60
    print(f"
{sep}")
    print(f"  {key.upper().replace(chr(95), chr(32))}")
    print(f"{sep}")
    print(f"  {value}")

## Residual Diagnostics

Bias, normality, autocorrelation, heteroscedasticity checks.

In [ ]:
from timesage.interpret.diagnostics import diagnose_residuals

diagnosis = diagnose_residuals(result_auto.residuals)
for test_name, result in diagnosis.items():
    print(f"
{test_name}:")
    if isinstance(result, dict):
        for k, v in result.items():
            print(f"   {k}: {v}")
    else:
        print(f"   {result}")

## Model Comparison - All Models Side by Side

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
models = ["arima", "ets", "theta", "rf", "xgboost", "lightgbm"]
results = {}
for ax, model_name in zip(axes.flat, models):
    try:
        result = series.forecast(horizon=12, model=model_name)
        results[model_name] = result
        plt.sca(ax)
        result.plot()
        ax.set_title(f"{model_name.upper()} (MAPE: {result.mape:.1f}%)", fontsize=12, fontweight="bold")
    except Exception as e:
        ax.set_title(f"{model_name.upper()} - Error")
        ax.text(0.5, 0.5, str(e)[:80], ha="center", va="center", fontsize=9)
plt.suptitle("Model Comparison - Airline Passengers", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### Metrics Comparison Table

In [ ]:
metrics_data = []
for name, res in results.items():
    metrics_data.append({
        "Model": name.upper(),
        "MAE": round(res.mae, 2),
        "RMSE": round(res.rmse, 2),
        "MAPE (%)": round(res.mape, 2),
        "R2": round(res.r2, 4),
        "MASE": round(res.mase, 4) if res.mase else None
    })
comparison_df = pd.DataFrame(metrics_data).sort_values("MAPE (%)")
print("Best model by MAPE:", comparison_df.iloc[0]["Model"])
comparison_df

## Custom Forecast Parameters

In [ ]:
result_custom = series.forecast(
    horizon=24,
    model="auto",
    test_size=0.2,
    confidence=0.95
)
result_custom.summary()

In [ ]:
result_custom.plot()
plt.title("Custom Forecast - 24 Months, 95% CI", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Feature Importance (ML Models)

In [ ]:
result_rf2 = series.forecast(horizon=12, model="rf")
if hasattr(result_rf2, "feature_importance") and result_rf2.feature_importance:
    print("Random Forest - Top Features:
")
    sorted_feats = sorted(result_rf2.feature_importance.items(), key=lambda x: x[1], reverse=True)
    for i, (name, score) in enumerate(sorted_feats[:15], 1):
        bar = chr(9608) * int(score * 50)
        print(f"  {i:2d}. {name:20s} {score:.4f} {bar}")
else:
    print("Feature importance not available")

## Other Built-in Datasets

In [ ]:
sunspots = ts.load_sunspots()
print(f"Sunspots: {sunspots.shape[0]} observations")
trend_data = ts.load_synthetic_trend()
seasonal_data = ts.load_synthetic_seasonal()
print(f"Synthetic Trend: {trend_data.shape[0]} observations")
print(f"Synthetic Seasonal: {seasonal_data.shape[0]} observations")

sun_series = ts.TimeSeries(sunspots, name="Sunspots")
plt.figure(figsize=(12, 4))
sun_series.plot()
plt.title("Sunspot Activity", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Color Palette

In [ ]:
from timesage.plot.theme import COLORS, PALETTE
print("TimeSage Color Palette:
")
for name, hex_code in COLORS.items():
    print(f"  {name:12s}  {hex_code}")
print(f"
Full palette: {PALETTE}")

---
## Summary

| Feature | Command |
|---|---|
| Load data | `ts.load_airline()` |
| TimeSeries | `ts.TimeSeries(data)` |
| Describe | `series.describe()` |
| Plot | `series.plot()` |
| Stationarity | `series.test_stationarity()` |
| Seasonality | `series.detect_seasonality()` |
| ACF/PACF | `series.plot_acf()` |
| Decompose | `series.decompose()` |
| Full EDA | `series.eda()` |
| Features | `series.create_features()` |
| ARIMA | `series.forecast(model="arima")` |
| ETS | `series.forecast(model="ets")` |
| Theta | `series.forecast(model="theta")` |
| Random Forest | `series.forecast(model="rf")` |
| XGBoost | `series.forecast(model="xgboost")` |
| LightGBM | `series.forecast(model="lightgbm")` |
| AutoML | `series.forecast(model="auto")` |
| Interpret | `result.interpret()` |
| Diagnostics | `diagnose_residuals()` |
| Themes | `ts.set_theme("dark")` |

---
**Made with [TimeSage](https://github.com/mlnjsh/timesage) | pip install timesage-ts[full]**